# Experiment 4 — Feature Pruning: Less Is More

More features is not always better — especially when the test set
contains item-outlet combinations the model has never seen. Features that correlate
with the target in training can be pure noise at test time if they're based on
memorised patterns rather than generalisable signals.

**7 features dropped (tested one at a time via LB-simulated validation):**

| Feature dropped | Why |
|---|---|
| `Item_Weight` | Near-zero correlation with sales; adds noise |
| `Item_Visibility` | Noisy after imputation; not stable across outlets |
| `Visibility_MeanRatio` | Derived from a noisy feature; amplifies the noise |
| `Item_Type` | 16 categories; redundant with `Item_Category` (3 categories) |
| `Outlet_Identifier` | Memorises which outlet ID has high sales in train — but test contains different items at the same outlets, so this memorisation hurts |
| `Is_Grocery` | Redundant — `Outlet_Type` already encodes this |
| `ItemType_OutletType` | High-cardinality interaction; overfits training patterns |

**Result:** 9-feature model generalises better than the 16-feature model.
OOF RMSE looked worse after dropping features — but LB improved.
This confirmed that OOF was measuring training fit, not generalisation.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

SEEDS = [42, 7, 13, 99, 2024, 123, 456, 17, 31, 37,
         53, 71, 83, 97, 101, 127, 131, 137, 149, 163]
KF = 5

# ── 1. Load ───────────────────────────────────────────────────────────────────
train = pd.read_csv('train_v9rqX0R.csv')
test  = pd.read_csv('test_AbJTz2l.csv')
print(f"Train: {train.shape} | Test: {test.shape}")

# ── 2. Clean ──────────────────────────────────────────────────────────────────
def clean(df, rw=None, rv=None):
    df = df.copy()
    df['Item_Fat_Content'] = df['Item_Fat_Content'].replace(
        {'LF':'Low Fat','low fat':'Low Fat','reg':'Regular'})
    df['Item_Category'] = df['Item_Identifier'].str[:2]
    df.loc[df['Item_Category']=='NC','Item_Fat_Content'] = 'Non-Consumable'
    if rw is None: rw = df.groupby('Item_Identifier')['Item_Weight'].mean()
    df['Item_Weight'] = df.apply(
        lambda r: rw.get(r['Item_Identifier'], np.nan)
        if pd.isna(r['Item_Weight']) else r['Item_Weight'], axis=1)
    df['Item_Weight'].fillna(df['Item_Weight'].median(), inplace=True)
    df['Outlet_Size'] = df.groupby('Outlet_Type')['Outlet_Size'].transform(
        lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Medium'))
    df['Outlet_Age']  = 2013 - df['Outlet_Establishment_Year']
    df['MRP_Cluster'] = pd.cut(df['Item_MRP'], bins=[0,70,130,200,270],
                                labels=[0,1,2,3]).astype(int)
    return df, rw, rv

tc, rw, rv = clean(train)
sc, _,  _  = clean(test, rw, rv)

# Outlet MRP mean from train only
om = tc.groupby('Outlet_Identifier')['Item_MRP'].mean()
tc['Outlet_MRP_Mean'] = tc['Outlet_Identifier'].map(om)
sc['Outlet_MRP_Mean'] = sc['Outlet_Identifier'].map(om)

# ── 3. Encode ─────────────────────────────────────────────────────────────────
for col in ['Item_Fat_Content','Outlet_Size','Outlet_Location_Type',
            'Outlet_Type','Item_Category']:
    le = LabelEncoder().fit(pd.concat([tc[col], sc[col]]).astype(str))
    tc[col] = le.transform(tc[col].astype(str))
    sc[col] = le.transform(sc[col].astype(str))

# ── 4. Features — selected via LB-simulated validation ───────────────────────
# Dropped vs v10: Item_Weight, Item_Visibility, Visibility_MeanRatio,
#                 Item_Type, Outlet_Identifier, Is_Grocery, ItemType_OutletType
# These were all memorising training patterns that don't generalise to test items
feats = [
    'Item_MRP',             # strongest signal by far
    'MRP_Cluster',          # nonlinear MRP bucketing
    'Outlet_MRP_Mean',      # outlet price level
    'Outlet_Type',          # grocery vs supermarket type
    'Outlet_Age',           # outlet maturity
    'Outlet_Size',          # outlet size
    'Outlet_Location_Type', # tier 1/2/3 city
    'Item_Category',        # FD / DR / NC
    'Item_Fat_Content',     # low fat / regular / non-consumable
]

for df in [tc, sc]:
    for f in feats: df[f] = df[f].fillna(0)

X      = tc[feats].values.astype(float)
y      = train['Item_Outlet_Sales'].values
X_test = sc[feats].values.astype(float)
print(f"Features: {len(feats)} — {feats}")

# ── 5. RF with 20 seeds, leaf=36 ─────────────────────────────────────────────
print(f"\n=== RF (leaf=36, {len(SEEDS)} seeds) ===")
all_oofs  = []
all_tests = []

for seed in SEEDS:
    kf  = KFold(n_splits=KF, shuffle=True, random_state=seed)
    oof = np.zeros(len(X))
    tp  = np.zeros(len(X_test))
    for tr_i, val_i in kf.split(X):
        m = RandomForestRegressor(
            n_estimators=500,
            min_samples_leaf=36,
            max_features=0.5,
            n_jobs=-1,
            random_state=seed
        )
        m.fit(X[tr_i], y[tr_i])
        oof[val_i] = m.predict(X[val_i])
        tp += m.predict(X_test) / KF

    rmse = np.sqrt(mean_squared_error(y, oof))
    print(f"  seed={seed:4d}  OOF={rmse:.2f}", flush=True)
    all_oofs.append(oof)
    all_tests.append(tp)

    # Checkpoint every 5 seeds
    if len(all_oofs) % 5 == 0:
        tp_so_far  = np.mean(all_tests, axis=0)
        oof_so_far = np.mean(all_oofs,  axis=0)
        sub = test[['Item_Identifier','Outlet_Identifier']].copy()
        sub['Item_Outlet_Sales'] = tp_so_far.clip(min=0)
        sub.to_csv(f'submission/predictions4_{len(all_oofs)}seeds.csv', index=False)
        print(f"  --> Checkpoint {len(all_oofs)} seeds: "
              f"OOF={np.sqrt(mean_squared_error(y, oof_so_far)):.2f}  "
              f"std={tp_so_far.std():.0f}")

# ── 6. Save final ─────────────────────────────────────────────────────────────
oof_final  = np.mean(all_oofs,  axis=0)
test_final = np.mean(all_tests, axis=0)
print(f"\nFinal {len(SEEDS)}-seed OOF: {np.sqrt(mean_squared_error(y, oof_final)):.2f}  "
      f"std={test_final.std():.0f}")

sub = test[['Item_Identifier','Outlet_Identifier']].copy()
sub['Item_Outlet_Sales'] = test_final.clip(min=0)
sub.to_csv('submission/predictions4.csv', index=False)

print(f"  mean={test_final.mean():.0f}  std={test_final.std():.0f}  "
      f"min={test_final.clip(min=0).min():.0f}")

Train: (8523, 12) | Test: (5681, 11)
Features: 9 — ['Item_MRP', 'MRP_Cluster', 'Outlet_MRP_Mean', 'Outlet_Type', 'Outlet_Age', 'Outlet_Size', 'Outlet_Location_Type', 'Item_Category', 'Item_Fat_Content']

=== RF (leaf=36, 20 seeds) ===
  seed=  42  OOF=1078.25
  seed=   7  OOF=1075.91
  seed=  13  OOF=1078.45
  seed=  99  OOF=1077.77
  seed=2024  OOF=1078.18
  --> Checkpoint 5 seeds: OOF=1076.66  std=1318
  seed= 123  OOF=1078.12
  seed= 456  OOF=1076.38
  seed=  17  OOF=1075.91
  seed=  31  OOF=1076.21
  seed=  37  OOF=1077.65
  --> Checkpoint 10 seeds: OOF=1076.13  std=1318
  seed=  53  OOF=1078.79
  seed=  71  OOF=1077.58
  seed=  83  OOF=1078.62
  seed=  97  OOF=1076.44
  seed= 101  OOF=1076.38
  --> Checkpoint 15 seeds: OOF=1076.17  std=1318
  seed= 127  OOF=1077.24
  seed= 131  OOF=1077.23
  seed= 137  OOF=1077.84
  seed= 149  OOF=1078.78
  seed= 163  OOF=1076.88
  --> Checkpoint 20 seeds: OOF=1076.20  std=1318

Final 20-seed OOF: 1076.20  std=1318
  mean=2181  std=1318  min=130
